#### 전력 사용량 전처리
출처: https://www.kepco.co.kr/home/customer/library/electricity-statistics/sales-volume/boardList.do<br>
설명: 지역별 + 용도 + 달 전력 사용량<br>
전처리 진행: 파일 필터링(각 연도별로 12월까지 저장된 파일만 가져오기) -> 파일별 결측, 공백, 컬럼 등 전처리<br>
문제: 2004, 2005 서울 없음 -> 제외 / ~2014년까지는 계약종별(주택용, 일반용 등)이 아닌 업종별<br>
TODO: 2014년 이전 데이터 업종별 -> 계약종별 매핑 / 2004, 2005 년도 제외<br>

In [1]:
import os
import re
import pandas as pd
import openpyxl


# =========================
# 1. 파일 필터링 함수
# =========================
def is_valid(file_name: str) -> bool:
    # 엑셀만 허용
    if not file_name.endswith((".xlsx", ".xls")):
        return False

    # -------------------------
    # 1) 2021 파일 처리
    # -------------------------
    # "2021" 또는 "21" 포함 파일 중
    # 오직 202112만 허용
    if ("2021" in file_name) or (file_name.startswith("21")):
        return "202112" in file_name

    # -------------------------
    # 2) 홈페이지 게시용 파일
    # -------------------------
    if "홈페이지" in file_name:
        m = re.search(r"(\d{6})", file_name)
        if not m:
            return False

        # YYYYMM에서 MM만 추출
        month = int(m.group(1)[4:])
        return month == 12

    # -------------------------
    # 3) 나머지 파일 허용
    # -------------------------
    return True


# =========================
# 2. 개별 파일 전처리
# =========================
def process_file(file_path: str, file_name: str) -> pd.DataFrame:

    df_raw = pd.read_excel(file_path, header=None)

    # "연도" 행 찾기
    header_row_idx = df_raw[df_raw.iloc[:, 0] == "연도"].index[0]

    # 컬럼 설정
    df_raw.columns = df_raw.iloc[header_row_idx]
    df = df_raw.iloc[header_row_idx + 1:].reset_index(drop=True)

    # 컬럼 정리
    df.columns = df.columns.astype(str).str.strip()

    # 월 컬럼
    months = ["1월","2월","3월","4월","5월","6월",
              "7월","8월","9월","10월","11월","12월"]

    # 숫자 변환
    df[months] = df[months].apply(pd.to_numeric, errors="coerce")

    # long format 변환
    df_long = df.melt(
        id_vars=["연도", "시도", "시군구", "계약종별"],
        value_vars=months,
        var_name="월",
        value_name="전력사용량"
    )

    # 서울시 필터
    df_long = df_long[df_long['시도'].str.startswith('서울')]

    # 월 정리
    df_long["월"] = df_long["월"].str.replace("월", "", regex=False).astype(int)

    # 연도에서 '년' 제거 + 숫자로 변환
    df_long["연도"] = (
        df_long["연도"]
        .astype(str)
        .str.replace("년", "", regex=False)
        .str.strip()
        .astype(int)
        )

    # 파일명 추가
    df_long["source_file"] = file_name

    return df_long


# =========================
# 3. 전체 폴더 처리 파이프라인
# =========================
def preprocess_kepco_folder(folder_path: str, output_path: str = None) -> pd.DataFrame:

    all_data = []

    files = [f for f in os.listdir(folder_path) if is_valid(f)]

    print(f"총 처리 파일 수: {len(files)}")

    for file in files:
        file_path = os.path.join(folder_path, file)

        print(f"처리 중: {file}")

        try:
            df_long = process_file(file_path, file)
            all_data.append(df_long)

        except Exception as e:
            print(f"❌ 오류 발생 ({file}): {e}")

    # 합치기
    result = pd.concat(all_data, ignore_index=True)
    result = result.sort_values(
        by=["연도", "시군구", "계약종별"],
        ascending=[True, True, True]
        ).reset_index(drop=True)

    print("전처리 완료")
    print("전체 데이터 크기:", result.shape)

    # =========================
    # 4. 저장 로직 추가
    # =========================
    if output_path is not None:

        # 폴더 자동 생성
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        if output_path.endswith(".csv"):
            result.to_csv(output_path, index=False, encoding="utf-8-sig")

        elif output_path.endswith(".xlsx"):
            result.to_excel(output_path, index=False)

        else:
            raise ValueError("지원하지 않는 파일 형식입니다 (csv, xlsx만 가능)")

        print(f"저장 완료: {output_path}")

    return result

In [ ]:
# from pathlib import Path

# BASE = Path.cwd().parent  # python 폴더 기준이면 parent

# folder_path = BASE / "data/extract/file"
# output_path = BASE / "data/output/kepco_result.csv"

In [11]:
# df = preprocess_kepco_folder(
#     folder_path=str(folder_path),
#     output_path=str(output_path)
# )